# 📈 02-机器学习 · 推导笔记 1：线性回归（手写全流程）

> 线性回归是**一切监督学习的起点**：闭式解（正规方程）、梯度下降、正则化三大技能全在这一篇。
> 核心实现全部手写（numpy 数值原语）；sklearn 仅用于对照验证（对照 cell 会显式标注）。

> 💡 一句话类比：中介给二手房估价，就是「面积 × 单价 + 楼龄 × 单价 + 地段 × 单价 + 基础价」——线性回归学到的正是每个特征该乘多少（权重 $w$），再加一个基础价（偏置 $b$）。

**运行环境**：Python 3.8+，仅依赖 numpy（对照 cell 需要 sklearn）。

## 📖 本章导读

> **这一章讲什么**：从「拟合直线」出发，推出最小二乘的**闭式解**（正规方程），再对比迭代法（梯度下降），最后用正则化（Ridge/Lasso）收尾——把「回归问题 = 优化问题」完整走一遍。

**学完本章，你应该能**：
1. 用一句话类比讲清线性回归（中介估价：每个特征一个单价 $w$ + 基础价 $b$）；
2. 手推正规方程 $w^*=(X^\top X)^{-1}X^\top y$，并说出它的前提（$X^\top X$ 可逆）；
3. 对比闭式解与梯度下降的取舍（矩阵求逆 $O(d^3)$ vs 迭代）；
4. 说清 L1/L2 正则的区别，以及 L1 为什么产生稀疏权重。

> 📌 全书第 1 章。阅读路径：本章导读 → 分节「原理 · 推导 · 代码」→ 自测清单 → 本章小结；统一框架见 00-总览。

## §1 模型假设与损失函数

**线性假设**：目标 $y$ 是特征 $x$ 的线性组合加噪声：

$$y = x^{\top} w + b + \varepsilon, \quad \varepsilon \sim \mathcal{N}(0, \sigma^2)$$

把偏置并入权重（在 $X$ 前加一列 1）：

$$\hat{y} = X w, \quad X \in \mathbb{R}^{n \times (d+1)}$$

**损失（MSE）**：

$$L(w) = \frac{1}{n} \| Xw - y \|_2^2 = \frac{1}{n} \sum_{i=1}^{n} (x_i^{\top} w - y_i)^2$$

**为什么用平方损失**：噪声高斯假设下，MLE 等价于最小化 MSE（推导见 §2）。

**两种求解路径**：

- **闭式解**（正规方程）：直接求导置零，$O(nd^2 + d^3)$
- **梯度下降**：迭代更新，$O(T \cdot nd)$，可在线/大规模

## §2 从 MLE 到最小二乘（概率视角推导）

假设 $y_i \sim \mathcal{N}(x_i^{\top} w, \sigma^2)$ 独立同分布，似然函数：

$$L(w) = \prod_{i=1}^n \frac{1}{\sqrt{2\pi\sigma^2}} \exp\left( -\frac{(y_i - x_i^{\top} w)^2}{2\sigma^2} \right)$$

取对数（最大化对数似然 = 最小化负对数似然）：

$$\log L(w) = -\frac{n}{2} \log(2\pi\sigma^2) - \frac{1}{2\sigma^2} \sum_{i=1}^n (y_i - x_i^{\top} w)^2$$

最大化 $\log L$ 等价于最小化 $\sum (y_i - x_i^{\top} w)^2$——**MLE = 最小二乘**（高斯噪声假设下）。

## §3 正规方程（闭式解）推导

对 $w$ 求梯度并置零：

$$\nabla_w L = \frac{2}{n} X^{\top}(Xw - y) = 0$$

$$\Rightarrow X^{\top}X w = X^{\top} y$$

$$\Rightarrow w^* = (X^{\top}X)^{-1} X^{\top} y$$

**可逆性**：$X^{\top}X$ 满秩才可逆（特征不共线、$n \ge d$）。退化时用伪逆 `np.linalg.pinv` 或加岭回归。

## §4 手写实现：闭式解 + 梯度下降

**手写原则**：

- 不调 `sklearn.linear_model`、不调 `np.linalg.lstsq` 之外的封装
- 标准化（z-score）用自己写的函数
- 每步打印关键中间量验证

**梯度下降更新公式**（批量）：

$$w_{t+1} = w_t - \eta \cdot \nabla L(w_t) = w_t - \eta \cdot \frac{2}{n} X^{\top}(Xw_t - y)$$

**反标准化（还原原始尺度）**：在标准化空间拟合得到 $y = w_0 + \sum_j w_j \frac{x_j - \mu_j}{s_j}$，展开还原回原始空间 $y = \sum_j w'_j x_j + b'$：

$$w'_j = \frac{w_j}{s_j}, \qquad b' = w_0 - \sum_{j} \frac{w_j \mu_j}{s_j}$$

> ⚠️ 还原**偏置**时必须保留 $-\sum_j w_j\mu_j/s_j$ 项。数据均值 $\mu \approx 0$ 时该项被掩盖（旧笔记本正是因此漏掉的），但均值非零时漏掉会明显算错——§4 代码附回归测试验证。

In [ ]:
import numpy as np

def make_data(n=100, d=2, noise=0.5, seed=42, w_true=None, bias=1.0):
    # 手写数据生成：默认 y = 3*x1 - 2*x2 + 1 + 噪声
    rng = np.random.default_rng(seed)
    X = rng.normal(0, 1, (n, d))
    if w_true is None:
        w_true = np.array([3.0, -2.0])
    y = X @ w_true + bias + rng.normal(0, noise, n)
    return X, y, w_true

def add_bias(X):
    return np.hstack([np.ones((len(X), 1)), X])

def standardize(X):
    mu = X.mean(axis=0)
    sd = X.std(axis=0)
    sd[sd == 0] = 1.0          # 防零除
    return (X - mu) / sd, mu, sd

def inverse_standardize(w_std, mu, sd):
    # 反标准化（还原原始尺度）：
    # 标准化空间: y = w_0 + Σ_j w_j (x_j - μ_j)/s_j
    # 展开:       y = Σ_j (w_j/s_j) x_j + (w_0 - Σ_j w_j μ_j / s_j)
    # 所以 w'_j = w_j/s_j；偏置 b' = w_0 - Σ_j w_j μ_j / s_j（必须保留均值项！）
    w_orig = w_std[1:] / sd
    b_orig = w_std[0] - np.sum(w_std[1:] * mu / sd)
    return w_orig, b_orig

def normal_eq(X, y):
    # 手写正规方程 w = (X^T X)^{-1} X^T y
    XtX = X.T @ X
    return np.linalg.inv(XtX) @ (X.T @ y)

def grad_descent(X, y, eta=0.1, epochs=500):
    n, d = X.shape
    w = np.zeros(d)
    for t in range(epochs):
        grad = (2.0 / n) * X.T @ (X @ w - y)   # 手写梯度
        w -= eta * grad
    return w

def mse(X, y, w):
    return float(np.mean((X @ w - y) ** 2))

# ---------- 演示 1：闭式解 vs 梯度下降（GD 解还原回原始尺度）----------
X, y, w_true = make_data()
Xb = add_bias(X)
Xz, mu, sd = standardize(X)
Xzb = add_bias(Xz)

w_closed = normal_eq(Xb, y)
w_gd = grad_descent(Xzb, y)
w_orig, b_orig = inverse_standardize(w_gd, mu, sd)   # 关键：还原 b 时减均值项
print('真实权重: [3.0, -2.0], 偏置 1.0')
print('正规方程解 w =', np.round(w_closed, 3))          # 期望约 [1.0, 2.9, -2.0]
print('梯度下降解(标准化后) w =', np.round(w_gd, 3))     # 标准化空间的解
print('GD 还原到原始尺度: 权重 =', np.round(w_orig, 3), ' 偏置 =', round(b_orig, 3))
print('闭式解 MSE =', round(mse(Xb, y, w_closed), 4))
print('GD 还原解 MSE =', round(mse(Xb, y, np.concatenate([[b_orig], w_orig])), 4))

# ---------- 回归测试：非零均值（μ≈3）下反标准化必须保留均值项 ----------
# 真实模型 y = 3*x1 - 2*x2 + 5；旧错误公式 b_orig = w_std[0]（漏均值项）会偏出 2+ 个单位
rng2 = np.random.default_rng(0)
X2 = rng2.normal(3.0, 2.0, (200, 2))                 # 均值≈3 的偏移数据
y2 = X2 @ np.array([3.0, -2.0]) + 5.0 + rng2.normal(0, 0.3, 200)

Xz2, mu2, sd2 = standardize(X2)
w_gd2 = grad_descent(add_bias(Xz2), y2)
w_orig2, b_orig2 = inverse_standardize(w_gd2, mu2, sd2)
print('\n[回归测试] y = 3*x1 - 2*x2 + 5（均值≈3 的非零偏移数据）')
print('还原权重 =', np.round(w_orig2, 3), ' 还原偏置 =', round(b_orig2, 3))
print('旧错误公式偏置（漏均值项） =', round(w_gd2[0], 3), ' → 明显偏大，验证均值项必须保留')
ok_restore = np.allclose(w_orig2, [3.0, -2.0], atol=0.15) and abs(b_orig2 - 5.0) < 0.2
print('断言：还原系数接近 (3, -2, 5) →', ok_restore)
assert ok_restore, '反标准化回归测试失败'
print('✓ 反标准化（保留均值项）验证通过')

## §4.5 sklearn 对照：手写解析解 vs LinearRegression

把 §3 推导的手写正规方程解与 `sklearn.linear_model.LinearRegression` 在同一数据上对比，断言系数一致（`allclose`）——验证手写实现与工业库完全等价。

In [ ]:
# ---- sklearn 对照：手写正规方程 vs LinearRegression（系数一致性）----
try:
    from sklearn.linear_model import LinearRegression
    HAS_SK = True
except ImportError:
    HAS_SK = False

if HAS_SK:
    rng = np.random.default_rng(11)
    Xs = rng.normal(0, 1, (150, 3))
    w_true = np.array([2.0, -1.5, 0.5])
    ys = Xs @ w_true + 3.0 + rng.normal(0, 0.2, 150)

    w_hand = normal_eq(add_bias(Xs), ys)          # 手写闭式解（§3）
    sk = LinearRegression().fit(Xs, ys)           # sklearn：默认带偏置 fit_intercept=True
    w_sk = np.concatenate([[sk.intercept_], sk.coef_])

    print('手写闭式解 w =', np.round(w_hand, 4))
    print('sklearn 解  w =', np.round(w_sk, 4))
    ok = np.allclose(w_hand, w_sk, atol=1e-8)
    print('allclose(w_hand, w_sk, atol=1e-8) =', ok)
    assert ok, '手写解析解与 sklearn 不一致！'
    print('✓ 手写正规方程与 sklearn.LinearRegression 系数完全一致')
else:
    print('未安装 sklearn，跳过对照（不影响手写部分）')

## §5 岭回归（L2 正则化）手写

**目标**：特征共线 / 高维时稳定解：

$$L(w) = \frac{1}{n}\|Xw - y\|^2 + \lambda \|w\|_2^2$$

**梯度**：

$$\nabla_w L = \frac{2}{n} X^{\top}(Xw - y) + 2\lambda w$$

**闭式解**：

$$w^* = (X^{\top}X + n\lambda I)^{-1} X^{\top} y$$

**效果**：把特征值小的方向压缩，防止过拟合；$\lambda \to 0$ 退化为普通最小二乘，$\lambda \to \infty$ 趋于 0。

In [ ]:
def ridge_closed(X, y, lam):
    n, d = X.shape
    return np.linalg.inv(X.T @ X + n * lam * np.eye(d)) @ (X.T @ y)

def ridge_gd(X, y, lam, eta=0.1, epochs=500):
    n, d = X.shape
    w = np.zeros(d)
    for _ in range(epochs):
        grad = (2.0 / n) * X.T @ (X @ w - y) + 2 * lam * w
        w -= eta * grad
    return w

# 构造共线数据演示稳定性（x2 几乎 = 2*x1）
rng = np.random.default_rng(1)
Xc = rng.normal(0, 1, (50, 2))
Xc[:, 1] = 2 * Xc[:, 0] + 0.001 * rng.normal(size=50)
yc = 3 * Xc[:, 0] + 0.5 * Xc[:, 1] + rng.normal(0, 0.1, 50)
Xcb = add_bias(Xc)

w_ols = normal_eq(Xcb, yc)
w_ridge = ridge_closed(Xcb, yc, lam=0.5)
print('共线下 OLS 权重(不稳定):', np.round(w_ols, 2))
print('共线下 Ridge 权重(稳定):', np.round(w_ridge, 2))
print('Ridge GD 与闭式一致性:', np.allclose(ridge_gd(Xcb, yc, 0.5), w_ridge, atol=1e-2))

## §6 L1（Lasso）与 L2 对比 + 手写 L1 梯度

**L1 惩罚**：$\lambda\|w\|_1$ 使权重**稀疏**（特征选择）；$\lambda\|w\|_2^2$ 使权重**收缩**但不归零。

**L1 的次梯度**（$w=0$ 处不可导）：

$$\frac{\partial}{\partial w_j} \lambda|w_j| = \lambda \cdot \operatorname{sign}(w_j) \ \ (w_j \ne 0)$$

**软阈值（坐标下降一步）**：

$$w_j = \operatorname{soft}\left(\rho_j, \lambda\right), \quad \operatorname{soft}(z, \lambda) = \operatorname{sign}(z)\max(|z| - \lambda, 0)$$

（完整 Lasso 用坐标下降最稳，此处给出梯度下降 + 次梯度近似便于理解。）

In [ ]:
def lasso_subgrad(X, y, lam, eta=0.05, epochs=1000):
    # 次梯度下降版 L1（示意；工程用坐标下降）
    n, d = X.shape
    w = np.zeros(d)
    for _ in range(epochs):
        grad_mse = (2.0 / n) * X.T @ (X @ w - y)
        grad_l1 = lam * np.sign(w)          # 次梯度：w=0 时取 0（简化）
        w -= eta * (grad_mse + grad_l1)
    return w

def soft_threshold(z, t):
    return np.sign(z) * np.maximum(np.abs(z) - t, 0)

# 稀疏数据：只有前 2 个特征有用
rng = np.random.default_rng(7)
Xs = rng.normal(0, 1, (80, 10))
ys = 2 * Xs[:, 0] - 1.5 * Xs[:, 1] + rng.normal(0, 0.3, 80)
Xsb = add_bias(Xs)

w_l1 = lasso_subgrad(Xsb, ys, lam=0.5)
w_l2 = ridge_closed(Xsb, ys, lam=0.5)
print('L1 权重（应稀疏，只有前3个非零）:', np.round(w_l1, 2))
print('L2 权重（整体收缩但不为零）:', np.round(w_l2, 2))
print('soft_threshold(2.5, 1.0) =', soft_threshold(2.5, 1.0), '，期望 1.5')
print('soft_threshold(-2.5, 1.0) =', soft_threshold(-2.5, 1.0), '，期望 -1.5')
print('soft_threshold(0.5, 1.0) =', soft_threshold(0.5, 1.0), '，期望 0.0')

## §7 多项式回归：过拟合演示（手写）

**多项式特征**：

$$\phi(x) = (1, x, x^2, \dots, x^p)$$

**风险**：$p$ 越大，训练 MSE 越低但测试 MSE 先降后升（**偏差-方差权衡**）：

$$\text{泛化误差} = \text{偏差}^2 + \text{方差} + \text{噪声}$$

In [ ]:
def poly_features(x, p):
    # 手写多项式特征矩阵 [1, x, x^2, ..., x^p]
    return np.vstack([x ** k for k in range(p + 1)]).T

rng = np.random.default_rng(3)
x = np.sort(rng.uniform(-3, 3, 40))
y = np.sin(x) + rng.normal(0, 0.2, 40)

results = []
for p in [1, 3, 9, 15]:
    Xp = poly_features(x, p)
    w = normal_eq(Xp, y)
    train_mse = mse(Xp, y, w)
    # 测试集（新噪声样本）
    y_test = np.sin(x) + rng.normal(0, 0.2, 40)
    test_mse = mse(Xp, y_test, w)
    results.append((p, round(train_mse, 4), round(test_mse, 4)))

print('  p  |  训练MSE  |  测试MSE  （演示过拟合曲线）')
for p, tr, te in results:
    print(f'  {p:2d} |  {tr:.4f}  |  {te:.4f}')
print('观察：p 增大训练误差下降，测试误差先降后升 → 过拟合')

## §8 常见 bug 与边界（面试踩坑区）

- **忘记加偏置列**：正规方程不包含截距时拟合的是过原点直线
- **不标准化导致梯度爆炸**：特征尺度差 1000 倍时 GD 发散——先 z-score
- **学习率过大/过小**：过大震荡发散、过小收敛慢；观察 loss 曲线判断
- **共线导致 $X^{\top}X$ 奇异**：`np.linalg.inv` 报 LinAlgError → 用岭回归或 pinv
- **`sd == 0` 零除**：常数特征标准化时除零——置 1 处理
- **反标准化漏均值项**：还原偏置忘减 $\sum_j w_j\mu_j/s_j$，数据均值非零时会明显算错（§4 已修复并附回归测试）
- **MSE 初始化**：梯度检查时用中心差分对比解析梯度（可加 `grad_check`）
- **L1 在 0 处不可导**：次梯度取 sign(0)=0 是简化近似，严格做法是软阈值/坐标下降

## §9 面试追问与扩展（加分项）

### 9.1 梯度检查（手写验证导数正确性）

$$g_{num} \approx \frac{L(w+\epsilon e_j) - L(w-\epsilon e_j)}{2\epsilon}, \quad \epsilon = 10^{-6}$$

数值梯度与解析梯度相对误差 < $10^{-5}$ 即正确——深度学习框架内部就是这么自检的。

### 9.2 最小二乘的统计性质（经典面试点）

- **无偏性**：$E[\hat{w}] = w^*$（高斯噪声下）
- 岭回归**有偏但方差小**——偏差-方差权衡的直接体现

### 9.3 与后续模型的关系

- 逻辑回归 = 线性模型 + sigmoid + 交叉熵（推导笔记 2）
- 广义线性模型：泊松回归、softmax 回归都是同一框架

## §10 自测清单（L1 必会 / L2 进阶 / L3 挑战）

### L1 必会（10 问）

- [ ] 1. 写出正规方程并推导 $w^* = (X^{\top}X)^{-1}X^{\top}y$
- [ ] 2. 写出 MSE 对 w 的梯度 $\frac{2}{n}X^{\top}(Xw-y)$
- [ ] 3. 手写闭式解（不调 sklearn）
- [ ] 4. 手写批量梯度下降 + 学习率调整
- [ ] 5. 说明为什么梯度下降前要标准化
- [ ] 6. 解释 $X^{\top}X$ 奇异的原因与对策
- [ ] 7. 写出岭回归闭式解与梯度
- [ ] 8. 区分 L1 稀疏与 L2 收缩
- [ ] 9. 手写 MSE / R² 指标
- [ ] 10. 多项式回归过拟合演示

### L2 进阶（10 问）

- [ ] 11. 从高斯 MLE 推导最小二乘
- [ ] 12. 手写随机梯度下降（SGD，逐样本更新）
- [ ] 13. 手写小批量梯度下降（mini-batch）
- [ ] 14. 梯度检查实现（中心差分）
- [ ] 15. 手写坐标下降版 Lasso（软阈值）
- [ ] 16. 推导偏差-方差分解公式
- [ ] 17. 交叉验证手写（k 折）
- [ ] 18. 岭回归的贝叶斯视角（高斯先验）
- [ ] 19. Lasso 的拉普拉斯先验视角
- [ ] 20. 手写加权最小二乘（WLS）

### L3 挑战（5 问）

- [ ] 21. 正规方程的数值稳定性（QR / SVD 分解求解）
- [ ] 22. 设计一个在线学习版本（流式数据）
- [ ] 23. 手写岭回归的留一交叉验证闭式（LOOCV trick）
- [ ] 24. 弹性网（ElasticNet = L1+L2）手写
- [ ] 25. 稳健回归（Huber 损失）手写

### 自测要点

- 10 分钟内盲写：闭式解 + 批量梯度下降（numpy）
- 能口述 MLE → 最小二乘的推导链路
- 能画偏差-方差权衡曲线并指出过拟合区

## 附录：参考与下节预告

- 《统计学习方法》李航 第 1 章（最小二乘）
- PRML 第 3 章（线性回归、偏差方差）

### 下节预告：推导笔记 2 — 逻辑回归（sigmoid / 交叉熵 / 梯度推导 / softmax 多分类）

> 💡 本笔记验收：`02-机器学习/README.md`「推导笔记 ≥ 6 篇」中 LR 篇打勾。

## 🏁 本章小结

- **原理一句话**：线性回归 = 最小二乘；最优解让**残差与特征空间正交**。
- **必会公式**：
  - 正规方程 $w^*=(X^\top X)^{-1}X^\top y$
  - 岭回归 $w^*=(X^\top X+\lambda I)^{-1}X^\top y$
  - 梯度下降 $w\leftarrow w-\eta X^\top(Xw-y)$
- **面试怎么考**：正规方程推导；$X^\top X$ 不可逆怎么办（→岭回归）；L1 vs L2（L1 为什么稀疏，菱形几何直觉）。
- **易错点**：特征不标准化；把 L1 当处处可导直接求导（实际用软阈值/子梯度）。

> 自测清单没把握的条目，回到对应小节重读后再打勾。